# 01 - 工具系统基础

本notebook介绍Agent工具系统的基本概念和使用方法。

## 学习目标
- 理解工具的设计原则和理论基础
- 掌握内置工具的使用方法
- 学会创建自定义工具
- 理解工具链和工具组合
- 掌握工具错误处理和重试机制

## 1. 理论背景

### Toolformer 论文核心思想

Toolformer (Schick et al., 2023) 提出了让LLM自主学习使用外部API的方法：

1. **Self-Supervised Learning**: 模型自己决定何时以及如何调用工具
2. **Minimal Intervention**: 只在必要时插入API调用
3. **Text Interface**: 所有工具通过文本输入输出交互

### 工具设计原则

- **单一职责**: 每个工具只做一件事
- **幂等性**: 相同输入产生相同输出
- **可组合性**: 工具可以串联使用
- **错误恢复**: 失败时有清晰的错误信息

In [ ]:
import sys
sys.path.insert(0, '../src')

from tools import (
    Tool, ToolConfig, ToolResult, ToolRegistry,
    CalculatorTool, SearchTool, PythonREPLTool,
    WikipediaTool, DateTimeTool, ToolError
)
import json
from typing import Dict, Any, List
from abc import ABC, abstractmethod

## 2. ToolResult 和 ToolConfig 详解

### ToolResult 结构

ToolResult封装了工具执行的完整结果：

In [ ]:
# 探索ToolResult的结构
calc = CalculatorTool()
result = calc.run(expression="2 + 3 * 4")

print(f"执行结果: {result.output}")
print(f"执行成功: {result.success}")
print(f"错误信息: {result.error}")
print(f"原始数据: {result.raw_data}")
print(f"元数据: {result.metadata}")
print(f"执行时长: {result.execution_time}s")

### ToolConfig 配置详解

In [ ]:
config = calc.config
print(f"工具名称: {config.name}")
print(f"工具描述: {config.description}")
print(f"参数定义: {json.dumps(config.parameters, indent=2, ensure_ascii=False)}")
print(f"必需参数: {config.required_params}")
print(f"返回类型: {config.return_type}")

## 3. 计算器工具深度使用

CalculatorTool使用AST安全解析数学表达式，防止代码注入攻击。

In [ ]:
calc = CalculatorTool()

# === 基本运算 ===
print("=== 基本运算 ===")
examples_basic = [
    "2 + 3 * 4",",
    "(2 + 3) * 4",
    "10 / 3",
    "10 // 3",
    "2 ** 8",
    "17 % 5"
]
for expr in examples_basic:
    result = calc.run(expression=expr)
    print(f"{expr:15} = {result.output}")

In [ ]:
# === 数学函数 ===
print("\n=== 数学函数 ===")
examples_math = [
    "sqrt(144)",
    "sqrt(2)",
    "sin(pi/2)",
    "cos(0)",
    "log(100, 10)",  # log base 10
    "log(e)",        # natural log
    "exp(2)",
    "abs(-5)",
    "round(3.14159, 2)",
    "ceil(3.2)",
    "floor(3.8)"
]
for expr in examples_math:
    result = calc.run(expression=expr)
    print(f"{expr:20} = {result.output}")

In [ ]:
# === 复杂表达式 ===
print("\n=== 复杂表达式 ===")
examples_complex = [
    "sin(pi/4) + cos(pi/4)",
    "sqrt(sin(pi/2)**2 + cos(pi/2)**2)",
    "(log(100) + log(1000)) / 2",
    "abs(-1 * exp(1) * sin(pi/2))"
]
for expr in examples_complex:
    result = calc.run(expression=expr)
    print(f"{expr:40} = {result.output}")

In [ ]:
# === 错误处理 ===
print("\n=== 错误处理 ===")
error_examples = [
    "1 / 0",           # 除零
    "sqrt(-1)",        # 负数开方
    "undefined_func()",# 未定义函数
    "import os",       # 非法表达式
    "__import__('os')" # 尝试注入
]
for expr in error_examples:
    result = calc.run(expression=expr)
    print(f"{expr:25} -> 成功: {result.success}, 错误: {result.error[:50] if result.error else 'None'}")

## 4. 搜索工具使用

SearchTool模拟网络搜索功能，演示了如何处理外部API。

In [ ]:
search = SearchTool()

# 基本搜索
result = search.run(query="Python机器学习")
print(f"查询: Python机器学习")
print(f"结果:\n{result.output}")

# 检查结果结构
print(f"\n结果数量: {len(result.raw_data) if result.raw_data else 0}")

In [ ]:
# 多个搜索查询
queries = [
    "深度学习框架对比",
    "Transformer架构详解",
    "LLM提示工程技巧"
]

for q in queries:
    result = search.run(query=q)
    print(f"\n查询: {q}")
    print(f"摘要: {result.output[:100]}...")

## 5. Python REPL 工具

在受限环境中安全执行Python代码，用于数据分析和计算任务。

In [ ]:
repl = PythonREPLTool()

# === 基本代码执行 ===
code1 = '''
# 数据处理
numbers = [1, 2, 3, 4, 5]
squared = [x**2 for x in numbers]
print(f"原始数据: {numbers}")
print(f"平方后: {squared}")
print(f"总和: {sum(squared)}")
'''
result1 = repl.run(code=code1)
print(result1.output)

In [ ]:
# === 数据分析 ===
code2 = '''
# 计算统计信息
data = [23, 45, 67, 89, 12, 34, 56, 78, 90, 11]
mean = sum(data) / len(data)
sorted_data = sorted(data)
median = sorted_data[len(data)//2]

print(f"数据: {data}")
print(f"均值: {mean:.2f}")
print(f"中位数: {median}")
print(f"最大值: {max(data)}")
print(f"最小值: {min(data)}")
'''
result2 = repl.run(code=code2)
print(result2.output)

In [ ]:
# === 字符串处理 ===
code3 = '''
text = "Hello, World! Python is great."
words = text.split()
word_count = len(words)
char_count = len(text.replace(" ", ""))
uppercase = text.upper()

print(f"原文: {text}")
print(f"单词数: {word_count}")
print(f"字符数: {char_count}")
print(f"大写: {uppercase}")
'''
result3 = repl.run(code=code3)
print(result3.output)

In [ ]:
# === 错误代码处理 ===
code_error = '''
# 这会产生一个错误
print(undefined_variable)
'''
result_error = repl.run(code=code_error)
print(f"执行成功: {result_error.success}")
print(f"错误信息: {result_error.error}")

## 6. 日期时间工具

In [ ]:
dt = DateTimeTool()

# 获取当前时间
result1 = dt.run(action="current")
print(f"当前时间: {result1.output}")

# 日期计算
result2 = dt.run(action="add", days=7, unit="days")
print(f"7天后: {result2.output}")

# 日期格式化
result3 = dt.run(action="format", format="%Y年%m月%d日")
print(f"格式化日期: {result3.output}")

## 7. 工具注册表 (ToolRegistry)

ToolRegistry管理多个工具，提供统一的查找和描述接口。

In [ ]:
registry = ToolRegistry()

# 注册多个工具
registry.register(CalculatorTool())
registry.register(SearchTool())
registry.register(PythonREPLTool())
registry.register(DateTimeTool())

print("=== 已注册工具 ===")
for tool_name in registry.list_tools():
    tool = registry.get_tool(tool_name)
    print(f"\n工具: {tool_name}")
    print(f"描述: {tool.config.description}")
    print(f"参数: {list(tool.config.parameters.keys())}")

In [ ]:
# 获取所有工具的描述（用于LLM提示）
tools_description = registry.get_tools_description()
print("\n=== 工具描述（用于LLM） ===")
print(tools_description)

In [ ]:
# 按名称获取工具
calc_tool = registry.get_tool("calculator")
if calc_tool:
    result = calc_tool.run(expression="5 * 6")
    print(f"通过注册表调用计算器: {result.output}")

# 尝试获取不存在的工具
missing = registry.get_tool("nonexistent")
print(f"\n不存在的工具: {missing}")

In [ ]:
# 注销工具
registry.unregister("calculator")
print(f"\n注销calculator后: {registry.list_tools()}")

# 重新注册
registry.register(CalculatorTool())
print(f"重新注册后: {registry.list_tools()}")

## 8. 创建自定义工具

通过继承Tool基类创建自定义工具。

In [ ]:
# === 示例1: 简单工具 ===
class UppercaseTool(Tool):
    """将文本转换为大写的工具"""
    
    @property
    def config(self) -> ToolConfig:
        return ToolConfig(
            name="uppercase",
            description="将输入的文本转换为大写字母",
            parameters={
                "text": {
                    "type": "string",
                    "description": "需要转换的文本内容"
                }
            },
            required_params=["text"]
        )
    
    def _run(self, text: str, **kwargs) -> str:
        return text.upper()

# 使用自定义工具
upper = UppercaseTool()
result = upper.run(text="hello world")
print(f"大写转换: {result.output}")

In [ ]:
# === 示例2: 带参数验证的工具 ===
class TemperatureConverterTool(Tool):
    """温度单位转换工具"""
    
    @property
    def config(self) -> ToolConfig:
        return ToolConfig(
            name="temperature_converter",
            description="在摄氏度和华氏度之间转换温度",
            parameters={
                "temperature": {
                    "type": "number",
                    "description": "温度数值"
                },
                "from_unit": {
                    "type": "string",
                    "description": "原始单位: C 或 F",
                    "enum": ["C", "F"]
                }
            },
            required_params=["temperature", "from_unit"]
        )
    
    def _run(self, temperature: float, from_unit: str, **kwargs) -> str:
        if from_unit.upper() == "C":
            # 摄氏度转华氏度
            fahrenheit = temperature * 9/5 + 32
            return f"{temperature}°C = {fahrenheit:.1f}°F"
        elif from_unit.upper() == "F":
            # 华氏度转摄氏度
            celsius = (temperature - 32) * 5/9
            return f"{temperature}°F = {celsius:.1f}°C"
        else:
            raise ToolError(f"不支持的单位: {from_unit}")

# 使用温度转换工具
temp_tool = TemperatureConverterTool()
print(temp_tool.run(temperature=25, from_unit="C").output)
print(temp_tool.run(temperature=77, from_unit="F").output)

In [ ]:
# === 示例3: 数据聚合工具 ===
class DataStatsTool(Tool):
    """数据统计分析工具"""
    
    @property
    def config(self) -> ToolConfig:
        return ToolConfig(
            name="data_stats",
            description="计算数值列表的统计信息",
            parameters={
                "data": {
                    "type": "array",
                    "description": "数值列表，JSON数组格式"
                },
                "stats": {
                    "type": "string",
                    "description": "统计类型: mean, median, mode, std, min, max, all",
                    "default": "all"
                }
            },
            required_params=["data"]
        )
    
    def _run(self, data: str, stats: str = "all", **kwargs) -> str:
        import json
        import statistics
        
        # 解析JSON数组
        try:
            numbers = json.loads(data)
            if not isinstance(numbers, list):
                raise ValueError("data必须是数组")
        except json.JSONDecodeError:
            raise ToolError("无效的JSON格式")

In [ ]:
        
        if not numbers:
            return "数据为空"
        
        results = {}
        
        if stats in ["all", "mean"]:
            results["均值"] = statistics.mean(numbers)
        if stats in ["all", "median"]:
            results["中位数"] = statistics.median(numbers)
        if stats in ["all", "stdev"]:
            results["标准差"] = statistics.stdev(numbers) if len(numbers) > 1 else 0
        if stats in ["all", "min"]:
            results["最小值"] = min(numbers)
        if stats in ["all", "max"]:
            results["最大值"] = max(numbers)
        if stats in ["all", "count"]:
            results["数量"] = len(numbers)
        
        # 格式化输出
        output_lines = ["=== 数据统计 ==="]
        for key, value in results.items():
            if isinstance(value, float):
                output_lines.append(f"{key}: {value:.4f}")
            else:
                output_lines.append(f"{key}: {value}")
        
        return "\n".join(output_lines)


In [ ]:
# 使用数据统计工具
stats = DataStatsTool()
data_json = json.dumps([23, 45, 67, 12, 89, 34, 56])
print(stats.run(data=data_json).output)

In [ ]:
# === 示例4: 带状态的工具 ===
class CounterTool(Tool):
    """计数器工具，维护内部状态"""
    
    def __init__(self):
        super().__init__()
        self._count = 0
    
    @property
    def config(self) -> ToolConfig:
        return ToolConfig(
            name="counter",
            description="计数器工具，支持增加、减少、重置和查询",
            parameters={
                "action": {
                    "type": "string",
                    "description": "操作类型: increment, decrement, reset, get",
                    "enum": ["increment", "decrement", "reset", "get"]
                },
                "value": {
                    "type": "number",
                    "description": "增加或减少的数值（可选）",
                    "default": 1
                }
            },
            required_params=["action"]
        )
    
    def _run(self, action: str, value: int = 1, **kwargs) -> str:
        if action == "increment":
            self._count += value
            return f"增加{value}，当前计数: {self._count}"
        elif action == "decrement":
            self._count -= value
            return f"减少{value}，当前计数: {self._count}"
        elif action == "reset":
            self._count = 0
            return f"计数器已重置，当前计数: {self._count}"
        elif action == "get":
            return f"当前计数: {self._count}"
        else:
            raise ToolError(f"未知操作: {action}")

# 使用带状态的工具
counter = CounterTool()
print(counter.run(action="increment").output)
print(counter.run(action="increment", value=5).output)
print(counter.run(action="decrement").output)
print(counter.run(action="get").output)
print(counter.run(action="reset").output)

## 9. 工具链和组合

多个工具可以组合使用，构建复杂的工作流。

In [ ]:
class ToolChain:
    """工具链，依次执行多个工具"""
    
    def __init__(self, tools: List[Tool]):
        self.tools = tools
        self.history = []
    
    def run(self, initial_input: str) -> str:
        current_output = initial_input
        results = []
        
        for i, tool in enumerate(self.tools):
            result = tool.run(input=current_output)
            results.append({
                "step": i + 1,
                "tool": tool.config.name,
                "input": current_output,
                "output": result.output,
                "success": result.success
            })
            current_output = result.output
        
        self.history = results
        return current_output
    
    def show_history(self):
        print("=== 工具链执行历史 ===")
        for step in self.history:
            print(f"\n步骤 {step['step']}: {step['tool']}")
            print(f"  输入: {step['input'][:50]}...")
            print(f"  输出: {step['output'][:50]}...")
            print(f"  状态: {'成功' if step['success'] else '失败'}")

# 创建工具链示例

In [ ]:
class InputValidatorTool(Tool):
    """输入验证工具"""
    @property
    def config(self) -> ToolConfig:
        return ToolConfig(
            name="validator",
            description="验证输入是否为有效数字",
            parameters={"input": {"type": "string"}},
            required_params=["input"]
        )
    def _run(self, input: str, **kwargs) -> str:
        try:
            float(input)
            return input
        except ValueError:
            raise ToolError(f"无效的数字输入: {input}")

# 构建工具链
chain = ToolChain([
    InputValidatorTool(),
    CalculatorTool(),
    UppercaseTool()
])

# 注意：这个示例中UppercaseTool可能不适合处理数字结果
# 实际应用中应根据需要选择合适的工具组合
print("工具链已创建，包含以下工具:")
for tool in chain.tools:
    print(f"  - {tool.config.name}: {tool.config.description}")

## 10. 工具错误处理和重试机制

In [ ]:
import time
from typing import Callable, Optional

class RetryWrapper:
    """为工具添加重试机制的包装器"""
    
    def __init__(
        self,
        tool: Tool,
        max_retries: int = 3,
        retry_on: Optional[Callable[[ToolResult], bool]] = None
    ):
        self.tool = tool
        self.max_retries = max_retries
        self.retry_on = retry_on or (lambda r: not r.success)
    
    @property
    def config(self) -> ToolConfig:
        return self.tool.config
    
    def run(self, **kwargs) -> ToolResult:
        last_result = None
        
        for attempt in range(self.max_retries + 1):
            result = self.tool.run(**kwargs)
            
            if not self.retry_on(result):
                # 不需要重试
                if attempt > 0:
                    result.metadata["retries"] = attempt
                return result
            
            last_result = result
            if attempt < self.max_retries:
                wait_time = 2 ** attempt  # 指数退避
                time.sleep(wait_time)
        
        # 所有重试都失败
        if last_result:
            last_result.metadata["retries"] = self.max_retries
        return last_result

# 使用重试包装器

In [ ]:
class UnreliableTool(Tool):
    """模拟不稳定的工具"""
    def __init__(self):
        super().__init__()
        self.attempts = 0
    
    @property
    def config(self) -> ToolConfig:
        return ToolConfig(
            name="unreliable",
            description="模拟不稳定的API",
            parameters={"input": {"type": "string"}},
            required_params=["input"]
        )
    
    def _run(self, input: str, **kwargs) -> str:
        self.attempts += 1
        if self.attempts < 3:
            raise ToolError(f"暂时错误 (尝试 {self.attempts})")
        return f"成功! 输入: {input}"

# 测试重试机制
unreliable = UnreliableTool()
retry_tool = RetryWrapper(unreliable, max_retries=3)
result = retry_tool.run(input="test")
print(f"结果: {result.output}")
print(f"重试次数: {result.metadata.get('retries', 0)}")

## 11. 工具性能监控

In [ ]:
import time
from collections import defaultdict

class ToolMonitor:
    """工具性能监控器"""
    
    def __init__(self):
        self.stats = defaultdict(lambda: {
            "calls": 0,
            "successes": 0,
            "failures": 0,
            "total_time": 0.0,
            "avg_time": 0.0
        })
    
    def track(self, tool_name: str, result: ToolResult, execution_time: float):
        stats = self.stats[tool_name]
        stats["calls"] += 1
        stats["total_time"] += execution_time
        stats["avg_time"] = stats["total_time"] / stats["calls"]
        if result.success:
            stats["successes"] += 1
        else:
            stats["failures"] += 1
    
    def report(self):
        print("=== 工具性能报告 ===")
        for tool_name, stats in self.stats.items():
            print(f"\n{tool_name}:")
            print(f"  调用次数: {stats['calls']}")
            print(f"  成功: {stats['successes']}")
            print(f"  失败: {stats['failures']}")
            print(f"  平均耗时: {stats['avg_time']:.4f}s")
            print(f"  成功率: {stats['successes']/stats['calls']*100:.1f}%" if stats['calls'] > 0 else "  成功率: N/A")

# 使用监控器
monitor = ToolMonitor()

# 模拟多次工具调用
calc = CalculatorTool()
test_expressions = ["2+2", "3*4", "10/2", "sqrt(16)", "2**8"]

for expr in test_expressions:
    start = time.time()
    result = calc.run(expression=expr)
    elapsed = time.time() - start
    monitor.track("calculator", result, elapsed)

# 显示性能报告
monitor.report()

## 12. 实战练习：构建数据分析工具链

结合所学知识，创建一个完整的数据分析工具链。

In [ ]:
# 定义数据分析工具
class DataParserTool(Tool):
    """解析CSV格式数据"""
    @property
    def config(self) -> ToolConfig:
        return ToolConfig(
            name="data_parser",
            description="解析逗号分隔的数值数据",
            parameters={"csv_data": {"type": "string"}},
            required_params=["csv_data"]
        )
    def _run(self, csv_data: str, **kwargs) -> str:
        rows = csv_data.strip().split('\n')
        data = []
        for row in rows:
            values = [float(x.strip()) for x in row.split(',') if x.strip()]
            data.extend(values)
        return str(data)

class DataAnalyzerTool(Tool):
    """分析数据"""
    @property
    def config(self) -> ToolConfig:
        return ToolConfig(
            name="data_analyzer",
            description="分析数值数据并生成报告",
            parameters={"data": {"type": "string"}},
            required_params=["data"]
        )
    def _run(self, data: str, **kwargs) -> str:
        import json
        numbers = json.loads(data)
        return f"数据数量: {len(numbers)}, 总和: {sum(numbers):.2f}, 平均值: {sum(numbers)/len(numbers):.2f}"

# 构建分析管道
parser = DataParserTool()
analyzer = DataAnalyzerTool()

# 测试完整流程
csv_input = """10, 20, 30
40, 50, 60
70, 80, 90"""

parsed = parser.run(csv_data=csv_input)
print(f"解析结果: {parsed.output}")

analyzed = analyzer.run(data=parsed.output)
print(f"分析结果: {analyzed.output}")

## 总结

本notebook介绍了Agent工具系统的完整知识体系：

### 核心概念
- **Tool**: 工具基类，定义了统一的接口
- **ToolConfig**: 工具配置，描述工具的元数据
- **ToolResult**: 执行结果，封装输出和状态
- **ToolRegistry**: 工具注册表，管理多个工具

### 内置工具
- CalculatorTool: 安全的数学计算
- SearchTool: 网络搜索模拟
- PythonREPLTool: 代码执行
- DateTimeTool: 日期时间操作

### 高级技巧
- 自定义工具创建
- 工具链组合
- 错误处理和重试
- 性能监控

### 最佳实践
1. 保持工具功能单一且明确
2. 提供清晰的参数描述
3. 处理所有可能的异常情况
4. 使用重试机制提高鲁棒性
5. 监控工具性能和使用情况